In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import matplotlib.pyplot as plt
import matplotlib as mpl
import platform
from scipy.stats import kstest
from scipy.stats import shapiro

In [2]:
profile_path = "../../data/raw/첨부파일 01_User_Profile.csv"
event_path = "../../data/raw/첨부파일 02_Event_Log.csv"

user_df = pd.read_csv(profile_path)
event_df = pd.read_csv(event_path)

|User_Profile.csv|

User_ID, 

가입일자(KST 기준), 

가입경로(오가닉/퍼포먼스광고), 

기기(iOS/Android), 

알림수신동의여부, 

알림수신동의_변경일자(KST 기준 / 변경 없는 경우 NULL)|

가입자 프로필 원천 데이터 (최소 10,000행 이상) — 날짜 컬럼은 KST 기준|

|Event_Log.csv|

|User_ID, Event_Time(타임스탬프), 

Event_Type(앱실행/수면기록/운동기록/마음챙김/식단기록/챌린지_탐색/챌린지참여/알림수신/알림오픈/온보딩_완료), 

Session_ID, 알림_유형(리마인드/챌린지_알림/광고성 — 알림 이벤트에만 값 존재)|

6개월치 앱 내 사용자 행동 로그 원천 데이터|

In [3]:
print("사용자 프로필 데이터 크기:", user_df.shape)
print("이벤트 로그 데이터 크기:", event_df.shape)

print("\n사용자 프로필 컬럼")
print(user_df.columns.tolist())

print("\n이벤트 로그 컬럼")
print(event_df.columns.tolist())

사용자 프로필 데이터 크기: (12500, 6)
이벤트 로그 데이터 크기: (1757262, 5)

사용자 프로필 컬럼
['User_ID', '가입일자', '가입경로', '기기', '알림수신동의여부', '알림수신동의_변경일자']

이벤트 로그 컬럼
['User_ID', 'Event_Time', 'Event_Type', 'Session_ID', '알림_유형']


In [4]:
user_df.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


In [5]:
event_df.head()

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


In [6]:
user_df.info()
event_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   User_ID      12500 non-null  str   
 1   가입일자         12500 non-null  str   
 2   가입경로         12363 non-null  str   
 3   기기           12379 non-null  str   
 4   알림수신동의여부     12384 non-null  object
 5   알림수신동의_변경일자  1976 non-null   str   
dtypes: object(1), str(5)
memory usage: 586.1+ KB
<class 'pandas.DataFrame'>
RangeIndex: 1757262 entries, 0 to 1757261
Data columns (total 5 columns):
 #   Column      Dtype
---  ------      -----
 0   User_ID     str  
 1   Event_Time  str  
 2   Event_Type  str  
 3   Session_ID  str  
 4   알림_유형       str  
dtypes: str(5)
memory usage: 67.0 MB


In [7]:
user_df.isna().sum().sort_values(ascending=False)

알림수신동의_변경일자    10524
가입경로             137
기기               121
알림수신동의여부         116
가입일자               0
User_ID            0
dtype: int64

In [8]:
event_df.isna().sum().sort_values(ascending=False)

알림_유형         1538380
Session_ID     241502
Event_Type      26456
User_ID             0
Event_Time          0
dtype: int64

## 전처리 시작

컬럼안에 있을 수 있는 공백 제거

In [9]:
user_string_cols = [
    "User_ID",
    "가입경로",
    "기기"
]

event_string_cols = [
    "User_ID",
    "Event_Type",
    "Session_ID",
    "알림_유형"
]


for col in user_string_cols:
    user_df[col] = user_df[col].str.strip()


for col in event_string_cols:
    event_df[col] = event_df[col].str.strip()

In [10]:
user_df.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


In [11]:
event_df.head()

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


결측치 확인

In [12]:
print('사용자 데이터 결측치')
print(
    user_df.isna().sum().sort_values(ascending=False)
)

print('이벤트 데이터 결측치')
print(
    event_df.isna().sum().sort_values(ascending=False)
)

사용자 데이터 결측치
알림수신동의_변경일자    10524
가입경로             137
기기               121
알림수신동의여부         116
가입일자               0
User_ID            0
dtype: int64
이벤트 데이터 결측치
알림_유형         1538380
Session_ID     241502
Event_Type      26456
User_ID             0
Event_Time          0
dtype: int64


결측 비율 확인

In [13]:
user_missing_rate = (
    user_df.isna().mean() * 100
).round(2)

event_missing_rate = (
    event_df.isna().mean() * 100
).round(2)

print("사용자 데이터 결측률")
print(user_missing_rate.sort_values(ascending=False))

print("\n이벤트 데이터 결측률")
print(event_missing_rate.sort_values(ascending=False))

사용자 데이터 결측률
알림수신동의_변경일자    84.19
가입경로            1.10
기기              0.97
알림수신동의여부        0.93
가입일자            0.00
User_ID         0.00
dtype: float64

이벤트 데이터 결측률
알림_유형         87.54
Session_ID    13.74
Event_Type     1.51
User_ID        0.00
Event_Time     0.00
dtype: float64


의견


알림수신 변경일자의 경우 결측이 생기면 그 뜻은 변경을 안했다는 뜻

-> 따라서 알람 수신이 true인 사람은 그 날짜 그대로 사용하고 False인 사람들 중에서 확인해보면 될 둣?

In [14]:
print("사용자 데이터 전체 중복 행:", user_df.duplicated().sum())
print("이벤트 데이터 전체 중복 행:", event_df.duplicated().sum())

print(
    "중복된 사용자 ID:",
    user_df.duplicated(subset=["User_ID"]).sum()
)

사용자 데이터 전체 중복 행: 0
이벤트 데이터 전체 중복 행: 0
중복된 사용자 ID: 0


날짜 타입변환

In [15]:
user_df["가입일자"] = pd.to_datetime(
    user_df["가입일자"],
    errors="coerce"
)

user_df["알림수신동의_변경일자"] = pd.to_datetime(
    user_df["알림수신동의_변경일자"],
    errors="coerce"
)

event_df["Event_Time"] = pd.to_datetime(
    event_df["Event_Time"],
    errors="coerce"
)

In [16]:
user_df["알림수신동의여부"].isna().sum()

np.int64(116)

알림수신동의 여부 값

In [17]:
notification_mapping = {
    True: "동의",
    False: "미동의",
}


user_df["알림수신동의여부"] = (
    user_df["알림수신동의여부"]
    .map(notification_mapping)
    .fillna("알수없음")
)

In [18]:
user_df["알림수신동의여부"].value_counts()

알림수신동의여부
동의      7984
미동의     4400
알수없음     116
Name: count, dtype: int64

의견 

알수없음은 알수 없음인지 앱상 오류인지 아니면 변경후 잘못된건지 확인해 봐야할 듯

이벤트 유형 결측값 처리

In [19]:
event_df["이벤트유형_결측여부"] = (
    event_df["Event_Type"].isna()
)

event_df["Event_Type"] = (
    event_df["Event_Type"]
    .fillna("알수없음")
)

In [20]:
event_df["이벤트유형_결측여부"].value_counts()

이벤트유형_결측여부
False    1730806
True       26456
Name: count, dtype: int64

In [21]:
event_df["Event_Type"].value_counts()

Event_Type
앱실행       728657
수면기록      242978
알림수신      194324
운동기록      131269
마음챙김      130344
식단기록      101366
챌린지참여      96829
챌린지_탐색     78101
알수없음       26456
알림오픈       21219
온보딩_완료      5719
Name: count, dtype: int64

의견

알수 없음이 사용한 적이 없어서 알수 없음인지 아니면 오류인지 확인해 봐야할듯

# 세션 ID 결측 확인

In [22]:
event_df["세션ID_결측여부"] = (
    event_df["Session_ID"].isna()
)

In [23]:
event_df["세션ID_결측여부"].value_counts()

세션ID_결측여부
False    1515760
True      241502
Name: count, dtype: int64

# 알림 유형 결측치 처리

In [24]:
event_df["알림_유형"] = (
    event_df["알림_유형"]
    .fillna("해당없음")
)

In [25]:
event_df["알림_유형"].value_counts()

알림_유형
해당없음      1538380
리마인드        85830
광고성         78262
챌린지_알림      54790
Name: count, dtype: int64

# 가입 시점 파생변수 생성

In [26]:
user_df["가입연월"] = (
    user_df["가입일자"]
    .dt.to_period("M")
    .astype(str)
)


In [27]:
user_df

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입연월
0,U0000001,2025-01-25,오가닉,iOS,동의,NaT,2025-01
1,U0000002,2025-05-06,오가닉,iOS,미동의,2025-05-24,2025-05
2,U0000003,2025-05-14,오가닉,iOS,미동의,NaT,2025-05
3,U0000004,2025-02-23,퍼포먼스광고,Android,동의,NaT,2025-02
4,U0000005,2025-02-18,퍼포먼스광고,Android,동의,NaT,2025-02
...,...,...,...,...,...,...,...
12495,U0012496,2025-04-24,오가닉,Android,미동의,NaT,2025-04
12496,U0012497,2025-01-28,오가닉,Android,미동의,NaT,2025-01
12497,U0012498,2025-02-05,오가닉,iOS,미동의,NaT,2025-02
12498,U0012499,2025-05-07,오가닉,Android,동의,NaT,2025-05


# 이벤트 날짜 파생변수 생성

In [28]:
event_df["이벤트일자"] = (
    event_df["Event_Time"]
    .dt.normalize()
)

event_df["이벤트연월"] = (
    event_df["Event_Time"]
    .dt.to_period("M")
    .astype(str)
)

In [29]:
event_df.head().T

,0,1,2,3,4
User_ID,U0000001,U0000001,U0000001,U0000001,U0000001
Event_Time,2025-01-25 07:25:45,2025-01-25 07:26:15,2025-01-25 07:26:55,2025-01-25 07:27:55,2025-01-25 20:30:00
Event_Type,앱실행,온보딩_완료,챌린지_탐색,챌린지참여,알림수신
Session_ID,2858201769,2858201769,2858201769,2858201769,NaN
알림_유형,해당없음,해당없음,해당없음,해당없음,광고성
이벤트유형_결측여부,False,False,False,False,False
세션ID_결측여부,False,False,False,False,True
이벤트일자,2025-01-25 00:00:00,2025-01-25 00:00:00,2025-01-25 00:00:00,2025-01-25 00:00:00,2025-01-25 00:00:00
이벤트연월,2025-01,2025-01,2025-01,2025-01,2025-01


# 사용자 프로필과 이벤트 로그 연결

In [30]:
event_df = event_df.merge(
    user_df[
        [
            "User_ID",
            "가입일자",
            "가입경로",
            "기기",
            "알림수신동의여부",
            "가입연월"
        ]
    ],
    on="User_ID",
    how="left",
    validate="many_to_one"
)

# 프로필에 없는 사용자 확인

In [31]:
missing_profile_user_count = (
    event_df.loc[
        event_df["가입일자"].isna(),
        "User_ID"
    ]
    .nunique()
)

print(
    "사용자 프로필에 없는 사용자 수:",
    missing_profile_user_count
)

사용자 프로필에 없는 사용자 수: 0


# 가입 후 경과일 계산

In [32]:
event_df["가입후경과일"] = (
    event_df["이벤트일자"]
    - event_df["가입일자"]
).dt.days

# 실제 사용자 활동 이벤트 정의

In [33]:
user_activity_events = [
    "앱실행",
    "온보딩_완료",
    "챌린지_탐색",
    "챌린지참여",
    "수면기록",
    "운동기록",
    "마음챙김",
    "식단기록",
    "알림오픈"
]

# 사용자 활동 여부 생성

In [34]:
event_df["사용자활동여부"] = (
    event_df["Event_Type"]
    .isin(user_activity_events)
)

# 알림수신 이벤트 여부 생성

In [35]:
event_df["알림수신이벤트여부"] = (
    event_df["Event_Type"] == "알림수신"
)

# 실제 활동이 있는 사용자 확인

In [36]:
activity_df = event_df[
    event_df["사용자활동여부"] == True
].copy()

active_users = activity_df["User_ID"].unique()

user_df["활동여부"] = (
    user_df["User_ID"]
    .isin(active_users)
)

# 사용자별 활동 요약

In [37]:
user_activity_summary = (
    activity_df
    .groupby("User_ID")
    .agg(
        최초활동일=("이벤트일자", "min"),
        최근활동일=("이벤트일자", "max"),
        활동일수=("이벤트일자", "nunique"),
        활동횟수=("Event_Type", "count")
    )
    .reset_index()
)

display(user_activity_summary.head())

,User_ID,최초활동일,최근활동일,활동일수,활동횟수
0,U0000001,2025-01-25,2025-04-24,80,432
1,U0000002,2025-05-06,2025-05-16,10,53
2,U0000003,2025-05-14,2025-05-14,1,3
3,U0000004,2025-02-23,2025-03-25,20,69
4,U0000005,2025-02-18,2025-05-18,82,357


# 가입 경로 전처리

In [38]:
user_df["가입경로"] = user_df["가입경로"].fillna("알수없음")

In [39]:
user_df["기기"] = user_df["기기"].fillna("알수없음")

In [40]:
user_df["User_ID"] = user_df["기기"].fillna("알수없음")

# 전처리 결과 확인

In [41]:
print("전처리 후 사용자 데이터 크기:", user_df.shape)
print("전처리 후 이벤트 데이터 크기:", event_df.shape)

display(user_df.head())
display(event_df.head())

전처리 후 사용자 데이터 크기: (12500, 8)
전처리 후 이벤트 데이터 크기: (1757262, 17)


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자,가입연월,활동여부
0,iOS,2025-01-25,오가닉,iOS,동의,NaT,2025-01,True
1,iOS,2025-05-06,오가닉,iOS,미동의,2025-05-24,2025-05,True
2,iOS,2025-05-14,오가닉,iOS,미동의,NaT,2025-05,True
3,Android,2025-02-23,퍼포먼스광고,Android,동의,NaT,2025-02,True
4,Android,2025-02-18,퍼포먼스광고,Android,동의,NaT,2025-02,True


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형,이벤트유형_결측여부,세션ID_결측여부,이벤트일자,이벤트연월,가입일자,가입경로,기기,알림수신동의여부,가입연월,가입후경과일,사용자활동여부,알림수신이벤트여부
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,해당없음,False,False,2025-01-25,2025-01,2025-01-25,오가닉,iOS,동의,2025-01,0,True,False
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,해당없음,False,False,2025-01-25,2025-01,2025-01-25,오가닉,iOS,동의,2025-01,0,True,False
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,해당없음,False,False,2025-01-25,2025-01,2025-01-25,오가닉,iOS,동의,2025-01,0,True,False
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,해당없음,False,False,2025-01-25,2025-01,2025-01-25,오가닉,iOS,동의,2025-01,0,True,False
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성,False,True,2025-01-25,2025-01,2025-01-25,오가닉,iOS,동의,2025-01,0,False,True


In [42]:
user_df.to_csv('../../data/preprocessed/user_profile.csv', index=False)
event_df.to_csv('../../data/preprocessed/event_log.csv', index=False)